In [2]:
# Imports
import polars as pl
import pandas as pd
import numpy as np
import json
import re

In [3]:
norm = pl.read_parquet('./data/normalized_data.parquet')

# Check for null or nan values in norm
norm.filter(pl.any_horizontal(pl.col(pl.Float64).is_null() | pl.col(pl.Float64).is_nan()))

# Remove nulls
norm = norm.filter(~pl.any_horizontal(pl.col(pl.Float64).is_null() | pl.col(pl.Float64).is_nan()))

# Making Polars DataFrame where each row is a pitch
    # (i.e. x_marker_frame, y_marker_frame, z_marker_frame are columns made from x, y, z, frame, and markerID columns of df)
def create_pitch_df(df):
    # Create a Polars DataFrame where each row is a pitch
    pitch_df = df.pivot(
        index = ["userID", "sessionID", "pitchNum", "height", "weight", "pitchType", "pitchSpeed", "p_throws"],
        on = ['markerID', 'frame_norm'],
        values = ['x', 'y', 'z']
    )
    
    # Pattern to match x_{"C7",0} or similar
    pattern = re.compile(r'([xyz])_\{"([^"]+)",(\d+)\}')

    # Change from naming convention of x_{"C7",0} to x_C7_0 
    for col in pitch_df.columns:
        if col.startswith('x_') or col.startswith('y_') or col.startswith('z_'):
            match = pattern.match(col)
            axis, marker, frame = match.groups()
            new_col_name = f"{axis}_{marker}_{frame}"
            pitch_df = pitch_df.rename({col: new_col_name})

    return pitch_df

# Create PCA-ready DataFrame
df = create_pitch_df(norm)
#df = pl.read_parquet('./data/pca_ready_data.parquet')
df.head()

userID,sessionID,pitchNum,height,weight,pitchType,pitchSpeed,p_throws,x_C7_0,x_C7_1,x_C7_2,x_C7_3,x_C7_4,x_C7_5,x_C7_6,x_C7_7,x_C7_8,x_C7_9,x_C7_10,x_C7_11,x_C7_12,x_C7_13,x_C7_14,x_C7_15,x_C7_16,x_C7_17,x_C7_18,x_C7_19,x_C7_20,x_C7_21,x_C7_22,x_C7_23,x_C7_24,x_C7_25,x_C7_26,x_C7_27,x_C7_28,…,z_T10_64,z_T10_65,z_T10_66,z_T10_67,z_T10_68,z_T10_69,z_T10_70,z_T10_71,z_T10_72,z_T10_73,z_T10_74,z_T10_75,z_T10_76,z_T10_77,z_T10_78,z_T10_79,z_T10_80,z_T10_81,z_T10_82,z_T10_83,z_T10_84,z_T10_85,z_T10_86,z_T10_87,z_T10_88,z_T10_89,z_T10_90,z_T10_91,z_T10_92,z_T10_93,z_T10_94,z_T10_95,z_T10_96,z_T10_97,z_T10_98,z_T10_99,z_T10_100
str,str,str,str,str,str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""001663""","""002979""","""002""","""79""","""224""","""FF""",84.9,"""R""",0.000894,-0.002095,-0.005044,-0.008621,-0.012187,-0.015632,-0.018199,-0.019736,-0.020734,-0.021373,-0.022053,-0.022319,-0.022395,-0.02238,-0.022116,-0.021415,-0.020305,-0.018988,-0.01717,-0.015117,-0.012702,-0.009775,-0.006859,-0.003446,0.000545,0.004559,0.008879,0.013713,0.018649,…,1.043039,1.019447,0.996764,0.975394,0.955043,0.938925,0.925863,0.914936,0.907453,0.902845,0.906859,0.924152,0.942343,0.964317,0.98969,1.01286,1.037042,1.067317,1.100145,1.128096,1.149072,1.161847,1.166601,1.166421,1.164158,1.162222,1.162095,1.164661,1.169859,1.178024,1.188346,1.200052,1.212393,1.222977,1.23271,1.240368,1.245988
"""001592""","""002815""","""005""","""75""","""214""","""FF""",89.6,"""R""",0.126997,0.121828,0.116415,0.111323,0.105744,0.100294,0.094618,0.089215,0.084609,0.080375,0.076557,0.072934,0.070157,0.067784,0.065847,0.064207,0.063362,0.062993,0.062367,0.061721,0.06097,0.060793,0.060783,0.060435,0.059839,0.059336,0.059201,0.058443,0.058004,…,1.083771,1.068037,1.051922,1.036903,1.020825,1.005626,0.990031,0.974639,0.960063,0.945186,0.931585,0.918385,0.906269,0.894801,0.884682,0.876335,0.868818,0.861577,0.853281,0.844776,0.834084,0.824472,0.819367,0.819622,0.827482,0.838531,0.849249,0.864497,0.880315,0.894153,0.911081,0.929625,0.953088,0.975924,0.995345,1.006016,1.009646
"""001606""","""002833""","""006""","""71""","""185""","""FF""",85.5,"""R""",0.191087,0.188155,0.184795,0.181236,0.177401,0.173563,0.169856,0.16634,0.163363,0.160904,0.159022,0.157571,0.156804,0.156696,0.156966,0.157615,0.158763,0.160295,0.161913,0.163603,0.16562,0.16782,0.170176,0.172415,0.174708,0.176945,0.179208,0.181457,0.183443,…,0.977591,0.965243,0.953593,0.942205,0.931444,0.921962,0.914428,0.907126,0.899475,0.891049,0.882012,0.871955,0.860913,0.849286,0.837982,0.829149,0.824092,0.822985,0.823776,0.827473,0.83567,0.850412,0.872408,0.896613,0.920356,0.941979,0.961852,0.978874,0.99595,1.010168,1.01868,1.024045,1.026535,1.027843,1.030689,1.034211,1.036295
"""001466""","""002653""","""006""","""75""","""170""","""FF""",87.7,"""R""",-0.122149,-0.116168,-0.109679,-0.103372,-0.097296,-0.09107,-0.085033,-0.078833,-0.072598,-0.066181,-0.059529,-0.052586,-0.045243,-0.038319,-0.030721,-0.023075,-0.015338,-0.008001,-0.000929,0.006419,0.013886,0.0214,0.028642,0.035511,0.042362,0.048881,0.055862,0.062677,0.069383,…,1.099852,1.09591,1.093203,1.091183,1.089124,1.085862,1.08023,1.071169,1.058899,1.045028,1.028319,1.011764,0.996294,0.982533,0.970604,0.962398,0.956669,0.955186,0.956015,0.959237,0.967039,0.980758,0.996727,1.013617,1.031975,1.055883,1.079698,1.096183,1.10609,1.113968,1.118291,1.121069,1.123552,1.126593,1.130946,1.136636,1.144225
"""001641""","""002923""","""008""","""70""","""162""","""FF""",76.3,"""R""",0.149857,0.147343,0.144785,0.141692,0.138732,0.135306,0.131694,0.128066,0.124044,0.120104,0.116221,0.112084,0.108068,0.104302,0.100617,0.097358,0.094404,0.092006,0.090112,0.088834,0.087882,0.087383,0.087129,0.087442,0.088122,0.089227,0.090562,0.092192,0.094358,…,0.913591,0.904274,

In [4]:
def convert_real_df_to_json(df: pl.DataFrame, output_filename="real_data.json"):
    # Ensure correct types
    df = df.with_columns([
        pl.col("frame_norm").cast(pl.Int64),
        pl.col("pitchNum").cast(pl.Int64),
        pl.col("x").cast(pl.Float64),
        pl.col("y").cast(pl.Float64),
        pl.col("z").cast(pl.Float64),
    ])

    # Group by pitch
    grouped = df.group_by(["userID", "sessionID", "pitchNum"]).agg([
        pl.first("pitchType").alias("pitchType"),
        pl.first("pitchSpeed").alias("pitchSpeed"),
        (pl.max("frame_norm") + 1).cast(pl.Int64).alias("max_frames"),
        pl.struct(["frame_norm", "markerID", "x", "y", "z"]).alias("points")
    ])

    pitches = []

    # Convert to JSON structure
    for row in grouped.iter_rows(named=True):
        frames = {}
        for p in row["points"]:
            f = int(p["frame_norm"])
            marker = p["markerID"]
            if f not in frames:
                frames[f] = {}
            frames[f][marker] = {
                "x": float(p["x"]),
                "y": float(p["y"]),
                "z": float(p["z"])
            }

        pitches.append({
            "id": f"{row['userID']}_{row['sessionID']}_{row['pitchNum']}",
            "label": f"U:{row['userID']} S:{row['sessionID']} P:{row['pitchNum']} "
                     f"({row['pitchType']} @ {row['pitchSpeed']}mph)",
            "max_frames": int(row["max_frames"]),
            "frames": frames
        })

    # Write JSON
    with open(output_filename, "w") as f:
        json.dump(pitches, f)

convert_real_df_to_json(norm)

In [5]:
# Filter rows where any value is null or NaN
df = df.filter(
    ~(
    pl.any_horizontal(
        # Check for missing data AND for floating point NaN
        pl.nth(range(8, len(df.columns))).is_null() | 
        pl.nth(range(8, len(df.columns))).is_nan()
    )
    )
)

df.head()

userID,sessionID,pitchNum,height,weight,pitchType,pitchSpeed,p_throws,x_C7_0,x_C7_1,x_C7_2,x_C7_3,x_C7_4,x_C7_5,x_C7_6,x_C7_7,x_C7_8,x_C7_9,x_C7_10,x_C7_11,x_C7_12,x_C7_13,x_C7_14,x_C7_15,x_C7_16,x_C7_17,x_C7_18,x_C7_19,x_C7_20,x_C7_21,x_C7_22,x_C7_23,x_C7_24,x_C7_25,x_C7_26,x_C7_27,x_C7_28,…,z_T10_64,z_T10_65,z_T10_66,z_T10_67,z_T10_68,z_T10_69,z_T10_70,z_T10_71,z_T10_72,z_T10_73,z_T10_74,z_T10_75,z_T10_76,z_T10_77,z_T10_78,z_T10_79,z_T10_80,z_T10_81,z_T10_82,z_T10_83,z_T10_84,z_T10_85,z_T10_86,z_T10_87,z_T10_88,z_T10_89,z_T10_90,z_T10_91,z_T10_92,z_T10_93,z_T10_94,z_T10_95,z_T10_96,z_T10_97,z_T10_98,z_T10_99,z_T10_100
str,str,str,str,str,str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""001663""","""002979""","""002""","""79""","""224""","""FF""",84.9,"""R""",0.000894,-0.002095,-0.005044,-0.008621,-0.012187,-0.015632,-0.018199,-0.019736,-0.020734,-0.021373,-0.022053,-0.022319,-0.022395,-0.02238,-0.022116,-0.021415,-0.020305,-0.018988,-0.01717,-0.015117,-0.012702,-0.009775,-0.006859,-0.003446,0.000545,0.004559,0.008879,0.013713,0.018649,…,1.043039,1.019447,0.996764,0.975394,0.955043,0.938925,0.925863,0.914936,0.907453,0.902845,0.906859,0.924152,0.942343,0.964317,0.98969,1.01286,1.037042,1.067317,1.100145,1.128096,1.149072,1.161847,1.166601,1.166421,1.164158,1.162222,1.162095,1.164661,1.169859,1.178024,1.188346,1.200052,1.212393,1.222977,1.23271,1.240368,1.245988
"""001592""","""002815""","""005""","""75""","""214""","""FF""",89.6,"""R""",0.126997,0.121828,0.116415,0.111323,0.105744,0.100294,0.094618,0.089215,0.084609,0.080375,0.076557,0.072934,0.070157,0.067784,0.065847,0.064207,0.063362,0.062993,0.062367,0.061721,0.06097,0.060793,0.060783,0.060435,0.059839,0.059336,0.059201,0.058443,0.058004,…,1.083771,1.068037,1.051922,1.036903,1.020825,1.005626,0.990031,0.974639,0.960063,0.945186,0.931585,0.918385,0.906269,0.894801,0.884682,0.876335,0.868818,0.861577,0.853281,0.844776,0.834084,0.824472,0.819367,0.819622,0.827482,0.838531,0.849249,0.864497,0.880315,0.894153,0.911081,0.929625,0.953088,0.975924,0.995345,1.006016,1.009646
"""001606""","""002833""","""006""","""71""","""185""","""FF""",85.5,"""R""",0.191087,0.188155,0.184795,0.181236,0.177401,0.173563,0.169856,0.16634,0.163363,0.160904,0.159022,0.157571,0.156804,0.156696,0.156966,0.157615,0.158763,0.160295,0.161913,0.163603,0.16562,0.16782,0.170176,0.172415,0.174708,0.176945,0.179208,0.181457,0.183443,…,0.977591,0.965243,0.953593,0.942205,0.931444,0.921962,0.914428,0.907126,0.899475,0.891049,0.882012,0.871955,0.860913,0.849286,0.837982,0.829149,0.824092,0.822985,0.823776,0.827473,0.83567,0.850412,0.872408,0.896613,0.920356,0.941979,0.961852,0.978874,0.99595,1.010168,1.01868,1.024045,1.026535,1.027843,1.030689,1.034211,1.036295
"""001466""","""002653""","""006""","""75""","""170""","""FF""",87.7,"""R""",-0.122149,-0.116168,-0.109679,-0.103372,-0.097296,-0.09107,-0.085033,-0.078833,-0.072598,-0.066181,-0.059529,-0.052586,-0.045243,-0.038319,-0.030721,-0.023075,-0.015338,-0.008001,-0.000929,0.006419,0.013886,0.0214,0.028642,0.035511,0.042362,0.048881,0.055862,0.062677,0.069383,…,1.099852,1.09591,1.093203,1.091183,1.089124,1.085862,1.08023,1.071169,1.058899,1.045028,1.028319,1.011764,0.996294,0.982533,0.970604,0.962398,0.956669,0.955186,0.956015,0.959237,0.967039,0.980758,0.996727,1.013617,1.031975,1.055883,1.079698,1.096183,1.10609,1.113968,1.118291,1.121069,1.123552,1.126593,1.130946,1.136636,1.144225
"""001641""","""002923""","""008""","""70""","""162""","""FF""",76.3,"""R""",0.149857,0.147343,0.144785,0.141692,0.138732,0.135306,0.131694,0.128066,0.124044,0.120104,0.116221,0.112084,0.108068,0.104302,0.100617,0.097358,0.094404,0.092006,0.090112,0.088834,0.087882,0.087383,0.087129,0.087442,0.088122,0.089227,0.090562,0.092192,0.094358,…,0.913591,0.904274,

In [55]:
# Taking transpose of the PCA-ready DataFrame to get the data in the correct format for PCA
    # (n_pitches, 3 * n_markers * n_frames)

X = df.select(pl.nth(range(8, len(df.columns)))).to_numpy().T

X_mean = np.mean(X, axis=1, keepdims = True)
X_centered = X - X_mean

cov_mat = (X_centered.T @ X_centered) / (X_centered.shape[1] - 1)
print(cov_mat.shape)

eigenvalues, eigenvectors = np.linalg.eigh(cov_mat)

idx = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

for i, eval, evec in zip(range(len(eigenvectors)), eigenvalues, eigenvectors.T):
    if i < 5:
        print(f"Eigenvalue: {np.round(eval,3)}, Eigenvector: {evec[:2]} ... [{evec[-1]}]")

(410, 410)
Eigenvalue: 112.782, Eigenvector: [ 0.03523557 -0.0756251 ] ... [0.09841981973225843]
Eigenvalue: 53.814, Eigenvector: [-0.08299333 -0.04061427] ... [0.06896826452784909]
Eigenvalue: 28.059, Eigenvector: [ 0.04161906 -0.05704452] ... [-0.05393539531386199]
Eigenvalue: 18.212, Eigenvector: [-0.0984126  -0.07559587] ... [0.03105824375360148]
Eigenvalue: 16.215, Eigenvector: [ 0.04602996 -0.04745273] ... [-0.07351174038448217]


In [56]:
# Calculate Explained Variance Ratio
total_variance = np.sum(eigenvalues)
explained_variance_ratio = eigenvalues / total_variance

# Calculate Cumulative Explained Variance
cumulative_variance = np.cumsum(explained_variance_ratio)

# Print results
for i, (eval, ratio) in enumerate(zip(eigenvalues, explained_variance_ratio)):
    print(f"PC{i+1}: Eigenvalue: {eval:.4f}, Variance Explained: {ratio:.2%}")
    if cumulative_variance[i] > 0.80:  # Stop showing once we hit 90%
        print(f"--- 80% variance reached at PC{i+1} ---")
        # Selecting top i principal components needed to get 90% of variance
        k = i + 1
        U = eigenvectors[:, :k]          # (M, k)
        Lambda = eigenvalues[:k]         # (k,)
        eps = 1e-12
        V = X_centered @ U / np.sqrt((X_centered.shape[1] - 1) * (Lambda + eps))
        break

PC1: Eigenvalue: 112.7825, Variance Explained: 33.60%
PC2: Eigenvalue: 53.8136, Variance Explained: 16.03%
PC3: Eigenvalue: 28.0585, Variance Explained: 8.36%
PC4: Eigenvalue: 18.2119, Variance Explained: 5.43%
PC5: Eigenvalue: 16.2154, Variance Explained: 4.83%
PC6: Eigenvalue: 12.0844, Variance Explained: 3.60%
PC7: Eigenvalue: 9.3025, Variance Explained: 2.77%
PC8: Eigenvalue: 7.6881, Variance Explained: 2.29%
PC9: Eigenvalue: 7.6153, Variance Explained: 2.27%
PC10: Eigenvalue: 5.4518, Variance Explained: 1.62%
--- 80% variance reached at PC10 ---


In [57]:
pca_export = {
    'X_mean': X_mean.squeeze().tolist(),          # (M,)
    'V': V.tolist(),                              # (M, k)
    'Lambda': Lambda.tolist(),                    # (k,)
    'feature_names': df.columns[8:],     # length M
    'marker_labels': norm.select(pl.col('markerID')).unique().to_series().to_list(),     # length n_markers
    'n_frames': 101,
}

with open('pca_data.json', 'w') as f:
    json.dump(pca_export, f)

In [58]:
def get_pca_scores_for_pitch(
    pitch_features,
    X_mean,
    V_components,
    Lambda
):
    """
    Returns variance-standardized PCA scores (z-scores).
    """
    pitch_vector = pitch_features.reshape(-1, 1)
    pitch_centered = pitch_vector - X_mean

    # Raw PCA coordinates
    raw_scores = V_components.T @ pitch_centered  # (k, 1)

    # Variance-standardized scores
    z_scores = raw_scores.flatten() / np.sqrt(Lambda)

    return z_scores



def compute_pca_scores_dataframe(df_pitches, X_mean, V_components, Lambda, n_training_samples):
    """
    Computes PCA scores for all pitches and returns as a dataframe.
    
    Args:
        df_pitches: Polars DataFrame with pitch data (from create_pitch_df)
        X_mean: global mean vector (shape: D, 1)
        V_components: principal components in feature space (shape: D, k)
        Lambda: eigenvalues (shape: k,)
        n_training_samples: number of training samples
    
    Returns:
        Polars DataFrame with columns:
        [userID, sessionID, pitchNum, pitchSpeed, PC1_Score, PC2_Score, ..., PC{k}_Score]
    """
    # Get feature columns (indices 8 onwards)
    feature_cols = df_pitches.columns[8:]
    metadata_cols = ["userID", "sessionID", "pitchNum", "pitchSpeed"]
    
    # Extract metadata
    metadata = df_pitches.select(metadata_cols)
    
    # Extract features as numpy array
    features_np = df_pitches.select(feature_cols).to_numpy()
    
    print(f"Computing PCA scores for {features_np.shape[0]} pitches...")
    
    # Compute PCA scores for all pitches
    pca_scores_list = []
    for i, pitch_features in enumerate(features_np):
        scores = get_pca_scores_for_pitch(pitch_features, X_mean, V_components, Lambda)

        pca_scores_list.append(scores)
        
        if (i + 1) % 50 == 0:
            print(f"  Computed scores for {i + 1}/{len(features_np)} pitches")
    
    # Convert scores to numpy array (n_pitches, k)
    pca_scores_array = np.array(pca_scores_list)
    
    # Create dataframe with PC columns
    k = pca_scores_array.shape[1]
    pc_column_names = [f"PC{i+1}_Score" for i in range(k)]
    
    # Convert to Polars DataFrame
    pc_df = pl.DataFrame({
        pc_column_names[i]: pca_scores_array[:, i].tolist() 
        for i in range(k)
    })
    
    # Combine metadata and PC scores
    result_df = pl.concat([metadata, pc_df], how="horizontal")
    
    print(f"Complete! Result shape: {result_df.shape}")
    print(f"Columns: {result_df.columns}")
    
    return result_df


# ============================================================================
# MAIN: Compute and save results
# ============================================================================

# Compute V_components (principal components in feature space)
eps = 1e-12
V_components = X_centered @ U / np.sqrt((X_centered.shape[1] - 1) * (Lambda + eps))
# V_components shape: (D, k)

print("Computing PCA scores for all pitches...")
pca_df = compute_pca_scores_dataframe(df, X_mean, V_components, Lambda, X_centered.shape[1])
pca_df.head()

Computing PCA scores for all pitches...
Computing PCA scores for 410 pitches...
  Computed scores for 50/410 pitches
  Computed scores for 100/410 pitches
  Computed scores for 150/410 pitches
  Computed scores for 200/410 pitches
  Computed scores for 250/410 pitches
  Computed scores for 300/410 pitches
  Computed scores for 350/410 pitches
  Computed scores for 400/410 pitches
Complete! Result shape: (410, 14)
Columns: ['userID', 'sessionID', 'pitchNum', 'pitchSpeed', 'PC1_Score', 'PC2_Score', 'PC3_Score', 'PC4_Score', 'PC5_Score', 'PC6_Score', 'PC7_Score', 'PC8_Score', 'PC9_Score', 'PC10_Score']


userID,sessionID,pitchNum,pitchSpeed,PC1_Score,PC2_Score,PC3_Score,PC4_Score,PC5_Score,PC6_Score,PC7_Score,PC8_Score,PC9_Score,PC10_Score
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""001663""","""002979""","""002""",84.9,0.712595,-1.678436,0.841693,-1.990272,0.930898,-0.659759,0.153786,-0.439434,0.054025,-1.2867
"""001592""","""002815""","""005""",89.6,-1.529423,-0.821373,-1.153654,-1.528832,-0.959672,-0.119241,0.384568,1.528992,-1.378017,0.121567
"""001606""","""002833""","""006""",85.5,-0.049933,0.162239,-0.316798,1.392576,0.176916,-0.342178,-0.539518,-0.104324,-0.770509,0.731388
"""001466""","""002653""","""006""",87.7,-0.381791,-1.196249,-1.520799,-0.836716,0.81291,1.009769,1.86348,0.129338,1.725385,0.204735
"""001641""","""002923""","""008""",76.3,-0.211277,-0.204575,0.881986,1.277992,0.180536,0.436721,0.980386,-0.22996,-0.083093,-1.157823


In [59]:
poi = pca_df.filter(
    (pl.col("userID") == "001000") & 
    (pl.col("sessionID") == "001718") & 
    (pl.col("pitchNum") == "006")
)
poi

userID,sessionID,pitchNum,pitchSpeed,PC1_Score,PC2_Score,PC3_Score,PC4_Score,PC5_Score,PC6_Score,PC7_Score,PC8_Score,PC9_Score,PC10_Score
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""001000""","""001718""","""006""",90.8,-0.869039,-0.632049,-0.035878,-0.19626,0.523956,0.221084,-0.612592,-1.075636,2.163856,0.227288
